In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze.sgs_series"
SILVER = f"{CATALOG}.silver.sgs_series"
QUARENTENA = f"{CATALOG}.silver.sgs_series_quarentena"

# 1. Tipagem segura: valores inválidos viram NULL em vez de quebrar o job
tipado = spark.table(BRONZE).select(
    F.expr("try_cast(serie_codigo AS INT)").alias("serie_codigo"),
    "serie_nome",
    F.expr("CAST(try_to_timestamp(data, 'dd/MM/yyyy') AS DATE)").alias("data_referencia"),
    F.expr("try_cast(valor AS DECIMAL(18,6))").alias("valor"),
    F.expr("try_cast(extraido_em AS TIMESTAMP)").alias("extraido_em"),
    "_arquivo_origem",
    "_ingerido_em",
)

# 2. Regras de qualidade
regra_valida = (
    F.col("serie_codigo").isNotNull()
    & F.col("data_referencia").isNotNull()
    & F.col("valor").isNotNull()
)
invalidos = tipado.filter(~regra_valida)
validos = tipado.filter(regra_valida)

# 3. Quarentena: guarda o que falhou, para investigar depois
invalidos.write.mode("overwrite").saveAsTable(QUARENTENA)

# 4. Deduplicação: uma linha por (série, data), ficando a extração mais recente
janela = Window.partitionBy("serie_codigo", "data_referencia").orderBy(
    F.col("extraido_em").desc(), F.col("_ingerido_em").desc()
)
dedup = (
    validos.withColumn("_rn", F.row_number().over(janela))
    .filter("_rn = 1")
    .drop("_rn")
    .withColumn("_atualizado_em", F.current_timestamp())
)

# 5. MERGE (upsert): insere o que é novo, atualiza o que foi revisado
if not spark.catalog.tableExists(SILVER):
    dedup.write.saveAsTable(SILVER)
else:
    (
        DeltaTable.forName(spark, SILVER).alias("t")
        .merge(
            dedup.alias("s"),
            "t.serie_codigo = s.serie_codigo AND t.data_referencia = s.data_referencia",
        )
        .whenMatchedUpdateAll(condition="s.extraido_em > t.extraido_em")
        .whenNotMatchedInsertAll()
        .execute()
    )

print(f"Válidos: {dedup.count()} | Quarentena: {invalidos.count()}")
display(spark.sql(f"""
    SELECT serie_nome, COUNT(*) AS registros,
           MIN(data_referencia) AS primeira, MAX(data_referencia) AS ultima
    FROM {SILVER} GROUP BY serie_nome
"""))

In [0]:
%sql
COMMENT ON TABLE workspace.silver.sgs_series IS 'Séries do SGS/BCB tipadas e deduplicadas: 1 linha por série e data';
ALTER TABLE workspace.silver.sgs_series ALTER COLUMN valor COMMENT 'Valor da série (R$ para dólar, % para Selic e IPCA)';
ALTER TABLE workspace.silver.sgs_series ALTER COLUMN data_referencia COMMENT 'Data de referência do valor';